In [1]:
import os
import sys
import torch
import string
from PIL import Image
from huggingface_hub import HfApi, login
from transformers import AutoModel, AutoProcessor, pipeline

sys.path.append(os.path.abspath(os.path.join("..")))
from deploy.captcha_crnn_base.configuration_captcha import CaptchaConfig
from deploy.captcha_crnn_base.processing_captcha import CaptchaProcessor
from deploy.captcha_crnn_base.modeling_captcha import CaptchaCRNN
from src.models.crnn.crnn_v1 import Captcha_CRNN_V1

# Deploy Captcha-CRNN-Base

In [3]:
# Load your old weights
old_weights = torch.load(Captcha_CRNN_V1.SAVE_DIR / "v4.pth", map_location="cpu")

# Initialize the new HF-style model
config = CaptchaConfig(num_chars=63)
hf_model = CaptchaCRNN(config)

# Load the weights into the HF-style model
hf_model.load_state_dict(old_weights)

# Save locally as a standard HF repo
hf_model.save_pretrained("../deploy/captcha_crnn_base")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
# Create the processor with your specific vocab
vocab = string.ascii_lowercase + string.ascii_uppercase + string.digits
processor = CaptchaProcessor(vocab=vocab)

# Save it into the SAME folder as your model
processor.save_pretrained("../deploy/captcha_crnn_base")

['../deploy/captcha_crnn_base/processor_config.json']

### Manually add to config.json

```
"auto_map": {
    "AutoConfig": "configuration_captcha.CaptchaConfig",
    "AutoModel": "modeling_captcha.CaptchaCRNN",
    "AutoProcessor": "processing_captcha.CaptchaProcessor"
  }
```

### Manually add to processor_config.json

```
"auto_map": {
    "AutoProcessor": "processing_captcha.CaptchaProcessor"
  }
```

### Manually add to config.json

```
"custom_pipelines": {
    "captcha-recognition": {
      "impl": "pipeline.CaptchaPipeline",
      "pt": ["AutoModel"],
      "type": "multimodal"
    }
  }
```

### Upload to HuggingFace

In [2]:
api = HfApi()
# login(token="***")

# 1. Create the repository
repo_id = "Graf-J/captcha-crnn-base"
api.create_repo(repo_id=repo_id, exist_ok=True)

# 2. Upload everything
api.upload_folder(
    folder_path="../deploy/captcha_crnn_base",
    repo_id=repo_id,
    commit_message="Add Live-Demo to Readme"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Graf-J/captcha-crnn-base/commit/23704b43dbf2a5d314eb2491adebc0436705afdc', commit_message='Add Live-Demo to Readme', commit_description='', oid='23704b43dbf2a5d314eb2491adebc0436705afdc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Graf-J/captcha-crnn-base', endpoint='https://huggingface.co', repo_type='model', repo_id='Graf-J/captcha-crnn-base'), pr_revision=None, pr_num=None)

### Test Code

In [6]:
from transformers import pipeline
from PIL import Image

# Initialize the pipeline
pipe = pipeline(
    task="captcha-recognition", 
    model="Graf-J/captcha-crnn-base", 
    trust_remote_code=True
)

# Load and predict
img = Image.open("Vb4cG.jpg")
result = pipe(img)
print(f"Decoded Text: {result['prediction']}")


config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

configuration_captcha.py:   0%|          | 0.00/232 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Graf-J/captcha-crnn-base:
- configuration_captcha.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pipeline.py:   0%|          | 0.00/561 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Graf-J/captcha-crnn-base:
- pipeline.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

Loading weights:   0%|          | 0/38 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

processing_captcha.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Graf-J/captcha-crnn-base:
- processing_captcha.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Decoded Text: Vb4cG


In [7]:
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor

# Load Model & Custom Processor
repo_id = "Graf-J/captcha-crnn-base"
processor = AutoProcessor.from_pretrained(repo_id, trust_remote_code=True)
model = AutoModel.from_pretrained(repo_id, trust_remote_code=True)

model.eval()

# Load and process image
img = Image.open("Vb4cG.jpg")
inputs = processor(img) 

# Inference
with torch.no_grad():
    outputs = model(inputs["pixel_values"])
    logits = outputs.logits

# Decode the prediction via CTC logic
prediction = processor.batch_decode(logits)[0]
print(f"Prediction: '{prediction}'")


Loading weights:   0%|          | 0/38 [00:00<?, ?it/s]

Prediction: 'Vb4cG'
